# Manuscript figures

Main text and SI figures. Reads the derived datasets in `data/derived/` (and the climate and geospatial inputs in `data/source/`) and writes figures to `results/figures/`. Where a figure reports numbers in the text, the plotted values are also written to `results/tables/`.

Same conventions as `src/analysis/manuscript_stats.ipynb`: historical 1991-2000, early future 2021-2050, late future 2051-2090; years where the BMP field or its baseline has zero yield are dropped.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy.stats import mannwhitneyu

DATA = "../../data/derived/"
SRC = "../../data/source/"
FIG = "../../results/figures/"
TAB = "../../results/tables/"
os.makedirs(FIG + "si/", exist_ok=True)
os.makedirs(FIG + "cbp/", exist_ok=True)
os.makedirs(TAB, exist_ok=True)

HIST = (1991, 2000)
EARLY = (2021, 2050)
LATE = (2051, 2090)
CROPS = ["corn", "soy", "wheat"]
BMPS = ["cc", "gb", "ma", "nm", "nt"]
BMP_COLOR = {"cc": "#648FFF", "gb": "#785EF0", "ma": "#DC267F", "nm": "#FE6100", "nt": "#FFB000"}
N_BOOT = 10000
SYN_FILES = {"corn": "synthetic_corn_results.csv", "soy": "synthetic_soy_results.csv", "wheat": "synthetic_wheat_results.csv"}

plt.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "DejaVu Sans"],
    "font.size": 8.5,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})


def load_annual(crops=CROPS):
    d = pd.read_csv(DATA + "all_annual_results.csv")
    d = d[d.crop.isin(crops)]
    hist = (d.scenario == "reference period") & d.year.between(*HIST)
    fut = (d.scenario != "reference period") & d.year.between(EARLY[0], LATE[1])
    d = d[hist | fut].replace([np.inf, -np.inf], np.nan)

    d["fail"] = (d.YLDG + d.YLDF) == 0
    d["ref"] = np.where(d.bmp.isin(["ma", "ma-base"]), "ma-base", "base")
    ref = d[d.bmp == d.ref][["scenario", "crop", "region", "year", "ref", "fail"]].rename(columns={"fail": "ref_fail"})
    d = d.merge(ref, on=["scenario", "crop", "region", "year", "ref"], how="left")
    d = d[~d.fail & ~d.ref_fail]
    return d.drop(columns=["fail", "ref", "ref_fail"])


def load_phase():
    d = pd.read_csv(DATA + "phase_re_dataset.csv")
    hist = (d.scenario == "historical") & d.phase_year.between(*HIST)
    fut = (d.scenario != "historical") & d.phase_year.between(EARLY[0], LATE[1])
    d = d[hist | fut]
    return d[(d.baseline_crop_state == "cropped") & (d.bmp_crop_state == "cropped")]


def load_storms():
    d = pd.read_csv(DATA + "storm_dataset.csv")
    return d[(d.baseline_crop_failure == 0) & (d.bmp_crop_failure == 0) & (d.baseline_TN > 0)]

## Study site

### Watershed map (Figure 1A)
State boundaries are downloaded from Natural Earth, so this cell needs an internet connection.

In [ ]:
import geopandas as gpd
from pyproj import Transformer

watershed = gpd.read_file(SRC + "geospatial/Chesapeake_Bay_Watershed_Boundary/Chesapeake_Bay_Watershed_Boundary.shp")
physio = gpd.read_file(SRC + "geospatial/physio_shp")
segments = gpd.read_file(SRC + "geospatial/CAST_land_segments/P6Beta_v3_LRSegs_081516.shp")
states = gpd.read_file("https://naciscdn.org/naturalearth/10m/cultural/ne_10m_admin_1_states_provinces.zip")

crs = "EPSG:5070"
watershed = watershed.to_crs(crs)
physio = physio.to_crs(crs)
segments = segments.to_crs(crs)
states = states.to_crs(crs)

provinces = ["APPALACHIAN PLATEAUS", "VALLEY AND RIDGE", "PIEDMONT", "COASTAL PLAIN", "BLUE RIDGE"]
physio = physio[physio.PROVINCE.isin(provinces)].dissolve(by="PROVINCE", as_index=False)
physio = gpd.clip(physio, watershed.union_all())

fips_region = {51001: "COASTAL PLAIN", 36017: "APPALACHIAN PLATEAUS", 24013: "PIEDMONT", 51820: "VALLEY AND RIDGE"}
site_labels = {51001: "Accomack, VA", 36017: "Chenango, NY", 24013: "Carroll, MD", 51820: "Waynesboro, VA"}
sites = segments[segments.FIPS.isin(fips_region)].dissolve(by="FIPS").reset_index()
sites["region"] = sites.FIPS.map(fips_region)

region_color = {"APPALACHIAN PLATEAUS": "#7A362A", "VALLEY AND RIDGE": "#C3A151", "PIEDMONT": "#5C4346", "COASTAL PLAIN": "#365E57"}
region_label = {"APPALACHIAN PLATEAUS": "APPALACHIAN\nPLATEAU", "VALLEY AND RIDGE": "VALLEY & RIDGE", "PIEDMONT": "PIEDMONT", "COASTAL PLAIN": "COASTAL\nPLAIN"}
region_rotation = {"APPALACHIAN PLATEAUS": 45, "VALLEY AND RIDGE": 45, "PIEDMONT": 45, "COASTAL PLAIN": 0}
region_xy = {"COASTAL PLAIN": (1750010, 1905000), "PIEDMONT": (1540000, 1800000), "VALLEY AND RIDGE": (1550000, 2075000), "APPALACHIAN PLATEAUS": (1575000, 2245000)}
ocean = "#A9CDE2"

xmin, ymin, xmax, ymax = watershed.total_bounds
xpad = (xmax - xmin) * 0.1
ypad = (ymax - ymin) * 0.1
x0, x1, y0, y1 = xmin - xpad, xmax + xpad, ymin - ypad, ymax + ypad

fig, ax = plt.subplots(figsize=(3.5, 5))
ax.set_facecolor(ocean)
ax.set_xlim(x0, x1)
ax.set_ylim(y0, y1)

states.plot(ax=ax, facecolor="#E8E8E8", edgecolor="#999999", linewidth=0.6, zorder=0)
watershed.plot(ax=ax, facecolor=ocean, edgecolor="black", linestyle="--", linewidth=1.0, zorder=1)
for region, color in region_color.items():
    physio[physio.PROVINCE == region].plot(ax=ax, color=color, edgecolor=None, alpha=0.9, zorder=2)
physio[physio.PROVINCE == "BLUE RIDGE"].plot(ax=ax, facecolor="white", edgecolor="#555555", hatch="////", linewidth=0.5, zorder=3)
for _, row in sites.iterrows():
    gpd.GeoSeries(row.geometry).plot(ax=ax, facecolor=region_color[row.region], edgecolor="black", linewidth=1.2, zorder=5)

for region, (x, y) in region_xy.items():
    ax.text(x, y, region_label[region], fontsize=8, fontweight="bold", ha="center", va="center", color="white",
            rotation=region_rotation[region], rotation_mode="anchor", zorder=6)

for _, row in states.cx[x0:x1, y0:y1].iterrows():
    abbrev = str(row.get("postal", "")).split(".")[-1]
    p = row.geometry.representative_point()
    if abbrev and abbrev not in ["PA", "MD", "DC"] and x0 <= p.x <= x1 and y0 <= p.y <= y1:
        ax.text(p.x, p.y, abbrev, fontsize=8, fontweight="bold", ha="center", va="center", color="#555555", alpha=0.85, zorder=4)

handles = [plt.Line2D([0], [0], marker="s", color="none", markerfacecolor=region_color[r.region], markeredgecolor="black",
                      markersize=7, markeredgewidth=0.8, label=site_labels[r.FIPS]) for _, r in sites.iterrows()]
handles.append(mpatches.Patch(facecolor="white", edgecolor="#555555", hatch="////", label="Blue Ridge Region"))
handles.append(plt.Line2D([0], [0], color="black", linestyle="--", linewidth=1.0, label="Watershed Boundary"))
ax.legend(handles=handles, loc="center", ncol=2, bbox_to_anchor=(0.5, -0.125), frameon=False, fontsize=7.5,
          columnspacing=1.5, handletextpad=0.4, borderpad=0.5)

bar_x = x0 + (x1 - x0) * 0.05
bar_y = y0 + (y1 - y0) * 0.025
tick = (y1 - y0) * 0.008
ax.plot([bar_x, bar_x + 100000], [bar_y, bar_y], color="black", linewidth=1.2, zorder=6)
for x in [bar_x, bar_x + 100000]:
    ax.plot([x, x], [bar_y - tick, bar_y + tick], color="black", linewidth=1.2, zorder=6)
ax.text(bar_x + 50000, bar_y + tick * 1.5, "100 km", fontsize=8, ha="center", va="bottom", fontweight="bold", zorder=6)

ax.annotate("N", xy=(0.93, 0.94), xycoords="axes fraction", fontsize=8, fontweight="bold", ha="center", va="bottom")
ax.annotate("", xy=(0.93, 0.94), xytext=(0.93, 0.88), xycoords="axes fraction", arrowprops=dict(arrowstyle="->", lw=1.0, color="black"))

ax.grid(True, color="#B0B0B0", linestyle="--", linewidth=0.5, zorder=0)
xt = [xmin - xpad * 0.5, (xmin + xmax) / 2, xmax + xpad * 0.5]
yt = [ymin - ypad * 0.5, (ymin + ymax) / 2, ymax + ypad * 0.5]
to_lonlat = Transformer.from_crs(crs, "EPSG:4326", always_xy=True)
ax.set_xticks(xt)
ax.set_yticks(yt)
ax.set_xticklabels([f"{abs(round(to_lonlat.transform(x, yt[1])[0], 1))}°W" for x in xt], fontsize=8)
ax.set_yticklabels([f"{round(to_lonlat.transform(xt[1], y)[1], 1)}°N" for y in yt], fontsize=8)

plt.savefig(FIG + "watershed_map.png", dpi=300, bbox_inches="tight")
plt.show()

### Projected climate change by region (Figure 1B-C)
Minimum, median, and maximum across the 31 GCMs (Bhatt et al. 2023), relative to 1995. Values are the 2011-2099 ranges provided by CBP.

In [ ]:
T = {
    "Appalachian\nPlateau": [1.16, 2.76, 4.35, 3.16, 5.17, 7.33],
    "Valley & Ridge": [1.06, 2.48, 3.82, 2.83, 4.70, 6.48],
    "Piedmont": [1.07, 2.58, 4.03, 2.85, 4.76, 6.78],
    "Coastal Plain": [0.83, 2.13, 3.29, 2.33, 4.00, 5.51],
}
P = {
    "Appalachian\nPlateau": [-4.98, 6.07, 16.60, 1.91, 9.63, 26.15],
    "Valley & Ridge": [-7.21, 7.58, 24.28, -7.80, 8.69, 29.08],
    "Piedmont": [-6.48, 7.22, 23.72, -6.70, 9.74, 37.79],
    "Coastal Plain": [-7.89, 6.65, 20.96, -8.89, 5.45, 35.51],
}
rcp_color = ["#2B5C8F", "#D95F02"]
regions = list(T)

fig, axes = plt.subplots(2, 1, figsize=(3, 5.0))
for ax, data, xlabel in zip(axes, [T, P], ["Temperature Change (°C)\nRelative to 1995", "Precipitation Change (%)\nRelative to 1995"]):
    for i, region in enumerate(regions):
        for j, offset in enumerate([-0.15, 0.15]):
            lo, med, hi = data[region][j * 3:(j + 1) * 3]
            ax.errorbar(med, i + offset, xerr=[[med - lo], [hi - med]], fmt="o", color=rcp_color[j],
                        elinewidth=1.1, capsize=2.0, capthick=1.0, ms=4.5, zorder=3)
    ax.set_xlabel(xlabel, labelpad=3)
    ax.axvline(0, color="#555555", lw=0.6, ls="--", zorder=1)
    ax.grid(axis="x", color="#E5E5E5", ls=":", lw=0.6, zorder=0)
    ax.spines[["top", "right", "left"]].set_visible(False)
    ax.spines["bottom"].set_linewidth(0.6)
    ax.tick_params(axis="y", length=0)
    ax.tick_params(axis="x", colors="#333333", length=3, width=0.6)
    ax.set_yticks(range(len(regions)))
    ax.set_yticklabels(regions)
    ax.invert_yaxis()

handles = [
    plt.Line2D([], [], color=rcp_color[0], marker="o", lw=0, ms=4.5, label="RCP 4.5"),
    plt.Line2D([], [], color=rcp_color[1], marker="o", lw=0, ms=4.5, label="RCP 8.5"),
    plt.Line2D([], [], color="#444444", marker="o", lw=0, ms=4.5, label="Median"),
    plt.Line2D([], [], color="#444444", marker="|", lw=1.1, ms=5, label="Range (31 GCMs)"),
]
fig.legend(handles=handles, loc="lower center", bbox_to_anchor=(0.55, -0.1), ncol=2, frameon=False,
           columnspacing=1.0, handletextpad=0.3, labelspacing=0.3)
plt.subplots_adjust(hspace=0.5, left=0.33, right=0.95, top=0.93, bottom=0.10)
plt.savefig(FIG + "climate_change_by_region.png", dpi=300, bbox_inches="tight")
plt.show()

### Climate deltas through time (SI)
Temperature and precipitation deltas across GCMs for each 30-year window, as boxplots and as median lines with the GCM range.

In [ ]:
df = pd.read_csv(SRC + "climate/future_deltas_summary.csv")
df["rcp"] = df.filename.str.extract(r"(RCP[48]5)")
df["midpoint"] = df.filename.str.extract(r"(\d{4})_(\d{4})").astype(int).mean(axis=1)

fips = ["N36017", "N51001", "N24013", "N51820"]
fips_label = {"N36017": "Appalachia (Chenango, NY)", "N51001": "Coastal Plain (Accomack, VA)",
              "N24013": "Piedmont (Carroll, MD)", "N51820": "Valley & Ridge (Waynesboro, VA)"}
rcp_color = {"RCP45": "#1f77b4", "RCP85": "#d62728"}
ylabel = {"temp": "Temperature Change (°C)", "precip": "Precipitation Change (%)"}

fig, axes = plt.subplots(2, 4, figsize=(14, 7), sharey="row")
for col, f in enumerate(fips):
    for row, var in enumerate(["temp", "precip"]):
        ax = axes[row, col]
        sub = df[(df.FIPS_NHL == f) & (df.variable == var)]
        mids = sorted(sub.midpoint.unique())
        data, pos, colors = [], [], []
        for i, mid in enumerate(mids):
            for j, rcp in enumerate(["RCP45", "RCP85"]):
                v = sub[(sub.midpoint == mid) & (sub.rcp == rcp)].Total.values
                if len(v):
                    data.append(v)
                    pos.append(i * 3 + j)
                    colors.append(rcp_color[rcp])
        bp = ax.boxplot(data, positions=pos, widths=0.8, patch_artist=True, showfliers=False)
        for m in bp["medians"]:
            m.set_color("black")
        for patch, c in zip(bp["boxes"], colors):
            patch.set_facecolor(c)
            patch.set_alpha(0.6)
        tops = [cap.get_ydata()[0] for cap in bp["caps"][1::2]]
        for k, p in enumerate(pos):
            ax.text(p, tops[k] + (0.09 if var == "temp" else 0.9), f"{np.median(data[k]):.1f}", ha="center", fontsize=6)
        ax.axhline(0, color="black", linestyle="--", linewidth=1, alpha=0.5)
        ax.set_xticks([i * 3 + 0.5 for i in range(len(mids))])
        ax.set_xticklabels([int(m) for m in mids], fontsize=8)
        if row == 0:
            ax.set_title(fips_label[f], fontsize=10)
        if col == 0:
            ax.set_ylabel(ylabel[var], fontsize=10)
axes[0, 0].legend(handles=[mpatches.Patch(facecolor=rcp_color[r], alpha=0.6, label=r.replace("RCP", "RCP ")) for r in rcp_color],
                  loc="upper left", fontsize=8)
plt.tight_layout()
plt.savefig(FIG + "si/climate_delta_boxplots.png", dpi=300, bbox_inches="tight")
plt.show()

fig, axes = plt.subplots(2, 4, figsize=(14, 7), sharey="row")
for col, f in enumerate(fips):
    for row, var in enumerate(["temp", "precip"]):
        ax = axes[row, col]
        for rcp, c in rcp_color.items():
            stats = df[(df.FIPS_NHL == f) & (df.variable == var) & (df.rcp == rcp)].groupby("midpoint").Total.agg(["min", "median", "max"])
            ax.plot(stats.index, stats["median"], color=c, label=rcp.replace("RCP", "RCP "), linewidth=2)
            ax.fill_between(stats.index, stats["min"], stats["max"], color=c, alpha=0.2)
        ax.axhline(0, color="black", linestyle="--", linewidth=1, alpha=0.5)
        if row == 0:
            ax.set_title(fips_label[f], fontsize=10)
        if col == 0:
            ax.set_ylabel(ylabel[var], fontsize=10)
axes[0, 0].legend(loc="upper left", fontsize=8)
plt.tight_layout()
plt.savefig(FIG + "si/climate_delta_timeseries.png", dpi=300, bbox_inches="tight")
plt.show()

## Sobol sensitivity (SI)
Climate + operations run, single parameters only. Bars are medians across region, crop, BMP, and output; whiskers are the median confidence interval.

In [ ]:
s = pd.read_csv(DATA + "sobol_climate_ops.csv")
s = s[s.crop.isin(CROPS) & ~s.parameter.str.contains(" x ")]

params = ["delta_p", "delta_t", "pec", "phu", "manure_incorp", "residue"]
param_label = {"delta_p": "Precip.\nChange", "delta_t": "Temp.\nChange", "pec": "Erosion\nControl", "phu": "Crop\nHeat Units",
               "manure_incorp": "Manure\nAdded", "residue": "Crop\nResidue"}
st = s.groupby("parameter").ST.median()[params]
st_conf = s.groupby("parameter").ST_conf.median()[params]
colors = ["#1f4e79" if p in ["delta_p", "delta_t"] else "#a9a9a9" for p in params]

fig, ax = plt.subplots(figsize=(6, 3))
ax.bar(range(len(params)), st, yerr=st_conf, color=colors, linewidth=0.7, error_kw={"elinewidth": 1, "capsize": 4})
ax.set_xticks(range(len(params)))
ax.set_xticklabels([param_label[p] for p in params], fontsize=10)
ax.set_ylabel("Median Total-Order Index (ST)", fontsize=11)
ax.set_ylim(0, st.max() * 1.15)
ax.spines[["top", "right"]].set_visible(False)
ax.grid(axis="y", alpha=0.3, linestyle=":")
ax.legend(handles=[mpatches.Patch(color="#1f4e79", label="Climate parameter"), mpatches.Patch(color="#a9a9a9", label="Operational parameter")],
          loc="upper right", frameon=False, fontsize=10)
plt.tight_layout()
plt.savefig(FIG + "si/sobol_parameter_dominance.png", dpi=300, bbox_inches="tight")
plt.show()

metrics = ["Qmm", "WYLDmm", "MUSLEt/ha", "YNkg/ha", "YPkg/ha", "QNkg/ha", "QPkg/ha"]
metric_label = ["Q", "WYLD", "Sediment", "YN", "YP", "QN", "QP"]
regions = ["a", "cp", "p", "vr"]
region_label = ["Appalachia", "Coastal Plain", "Piedmont", "Valley & Ridge"]

fig, axes = plt.subplots(1, 2, figsize=(6, 2.5), sharey=True)
for ax, param, title in zip(axes, ["delta_p", "delta_t"], ["ΔP", "ΔT"]):
    mat = s[s.parameter == param].groupby(["metric", "region"]).S1.median().unstack().reindex(index=metrics, columns=regions)
    im = ax.imshow(mat.values, cmap="YlGnBu", aspect="auto", vmin=0, vmax=1)
    ax.set_xticks(range(len(regions)))
    ax.set_xticklabels(region_label, rotation=30, ha="right", fontsize=10)
    ax.set_yticks(range(len(metrics)))
    ax.set_yticklabels(metric_label, fontsize=10)
    ax.set_title(f"S1 ({title})", fontsize=12)
    for i in range(len(metrics)):
        for j in range(len(regions)):
            v = mat.iloc[i, j]
            if not np.isnan(v):
                ax.text(j, i, f"{v:.2f}", ha="center", va="center", color="white" if v > 0.5 else "black", fontsize=10)
fig.colorbar(im, ax=axes, shrink=0.8, pad=0.02).set_label("Median First-Order Index (S1)", fontsize=10)
plt.savefig(FIG + "si/sobol_S1_heatmap.png", dpi=300, bbox_inches="tight")
plt.show()

fig, ax = plt.subplots(figsize=(6, 4))
x = np.arange(len(regions))
for k, (param, label, c) in enumerate([("delta_p", "ΔP", "#1f4e79"), ("delta_t", "ΔT", "#c0504d")]):
    g = s[s.parameter == param].groupby("region")
    ax.bar(x + (k - 0.5) * 0.35, g.S1.median()[regions], 0.35, yerr=g.S1_conf.median()[regions], label=label, color=c,
           edgecolor="black", linewidth=0.7, error_kw={"elinewidth": 1, "capsize": 3})
ax.set_xticks(x)
ax.set_xticklabels(region_label, fontsize=11)
ax.set_ylabel("Median First-Order Index (S1)", fontsize=11)
ax.legend(frameon=False, fontsize=10)
ax.spines[["top", "right"]].set_visible(False)
ax.grid(axis="y", alpha=0.3, linestyle=":")
plt.tight_layout()
plt.savefig(FIG + "si/sobol_climate_by_region.png", dpi=300, bbox_inches="tight")
plt.show()

## Annual BMP performance (Figure 3)
Top: baseline loads. Bottom: removal efficiency. Annual values are the median across crops, smoothed with an 11-year rolling median per region. Lines are the median across regions for each RCP, the band is the range across regions and both RCPs, and points are the historical median. RCP 8.5 is drawn from 2051, since both pathways share the same weather before then.

In [ ]:
d = load_annual()
hist = d[d.scenario == "reference period"]
fut = d[d.scenario != "reference period"]

re_metrics = ["RE_TN", "RE_TP", "RE_MUSL"]
loads = ["TN", "TP", "MUSL"]
load_label = {"TN": "TN (kg/ha)", "TP": "TP (kg/ha)", "MUSL": "Sediment (t/ha)"}
load_color = {"TN": "#4C72B0", "TP": "#CC6600", "MUSL": "#6B3A2A"}
bmp_label = {"cc": "Cover crop", "gb": "Grass buffer", "ma": "Manure incorp", "nm": "Nutrient mgmt", "nt": "No-till"}

re_year = fut[fut.bmp.isin(BMPS)].groupby(["scenario", "region", "bmp", "year"])[re_metrics].median().reset_index()
re_hist = hist[hist.bmp.isin(BMPS)].groupby(["region", "bmp", "year"])[re_metrics].median().reset_index()
load_year = fut[fut.bmp == "base"].groupby(["scenario", "region", "year"])[loads].median().reset_index()
load_hist = hist[hist.bmp == "base"].groupby(["region", "year"])[loads].median().reset_index()

for df, groups, cols in [(re_year, ["scenario", "region", "bmp"], re_metrics), (load_year, ["scenario", "region"], loads)]:
    df.sort_values(groups + ["year"], inplace=True)
    for m in cols:
        df[m + "_smooth"] = df.groupby(groups)[m].transform(lambda v: v.rolling(11, center=True, min_periods=11).median())


def draw(ax, df, col, color):
    band = df.groupby("year")[col].agg(["min", "max"])
    ax.fill_between(band.index, band["min"], band["max"], color=color, alpha=0.15, linewidth=0, zorder=2)
    for sc, ls, start in [("RCP 45", "-", EARLY[0]), ("RCP 85", "--", LATE[0])]:
        med = df[(df.scenario == sc) & (df.year >= start)].groupby("year")[col].median()
        ax.plot(med.index, med.values, color=color, linestyle=ls, linewidth=2.2, zorder=3)


fig, axes = plt.subplots(2, 3, figsize=(12, 8), sharex=True)
for i, m in enumerate(loads):
    ax = axes[0, i]
    draw(ax, load_year, m + "_smooth", load_color[m])
    ax.plot(2017, load_hist.groupby("year")[m].median().median(), marker="o", markersize=8, color=load_color[m],
            markeredgecolor="white", markeredgewidth=0.8, zorder=4)
    ax.set_title(load_label[m], fontsize=22)

for i, m in enumerate(re_metrics):
    ax = axes[1, i]
    ax.axhline(0, color="gray", linewidth=0.5, linestyle=":", zorder=0)
    for bm in BMPS:
        draw(ax, re_year[re_year.bmp == bm], m + "_smooth", BMP_COLOR[bm])
        ax.plot(2017, re_hist[re_hist.bmp == bm][m].median(), marker="o", markersize=8, color=BMP_COLOR[bm],
                markeredgecolor="white", markeredgewidth=0.8, zorder=4)

axes[0, 0].set_ylabel("Baseline Load", fontsize=22)
axes[1, 0].set_ylabel("Removal Efficiency (%)", fontsize=22)
for ax in axes.flat:
    ax.spines[["top", "right"]].set_visible(False)
    ax.tick_params(labelsize=18)
    ax.set_xlim(2013, 2092)

bmp_handles = [plt.Line2D([0], [0], color=BMP_COLOR[b], linewidth=2.5, label=bmp_label[b]) for b in BMPS]
style_handles = [
    plt.Line2D([0], [0], color="gray", linestyle="-", linewidth=2, label="RCP 4.5 median"),
    plt.Line2D([0], [0], color="gray", linestyle="--", linewidth=2, label="RCP 8.5 median"),
    plt.Line2D([0], [0], marker="o", markersize=8, linestyle="", color="gray", markeredgecolor="white", label="Historical median"),
    mpatches.Patch(color="gray", alpha=0.15, label="Regional range (RCP 4.5 & 8.5)"),
]
fig.add_artist(fig.legend(handles=bmp_handles, loc="upper center", ncol=5, frameon=False, bbox_to_anchor=(0.5, 0.02), fontsize=16))
fig.legend(handles=style_handles, loc="upper center", ncol=4, frameon=False, bbox_to_anchor=(0.5, -0.05), fontsize=16)
plt.tight_layout()
plt.savefig(FIG + "annual_bmp_performance.png", dpi=300, bbox_inches="tight")
plt.show()

## Crop-cycle phase

### Pre-plant to post-harvest change (SI)
Within-year difference in TN removal efficiency between post-harvest and pre-plant, all periods. Points are medians, lines are bootstrapped 95% CIs.

In [ ]:
d = load_phase()
d = d[np.isfinite(d.re_TN)]
rng = np.random.default_rng(0)

order = ["nt", "gb", "ma", "cc", "nm"]
label = {"cc": "Cover crop", "gb": "Grass buffer", "ma": "Manure\nIncorporation", "nm": "Nutrient\nManagement", "nt": "No-Till"}
marker = {"corn": "o", "soy": "s", "wheat": "^"}

p = d.pivot_table(index=["crop", "region", "scenario", "bmp", "phase_year"], columns="phase", values="re_TN").reset_index()
p["diff"] = p.postharvest - p.preplant

fig, ax = plt.subplots(figsize=(6.5, 3.5))
ax.axvline(0, color="#444444", linewidth=1.0, zorder=1)
rows = []
for i, bm in enumerate(order):
    y = len(order) - 1 - i
    for off, cr in zip([0.2, 0, -0.2], CROPS):
        v = p[(p.bmp == bm) & (p.crop == cr)]["diff"].dropna().values
        if len(v) == 0:
            continue
        lo, hi = np.percentile(np.median(v[rng.integers(0, len(v), (N_BOOT, len(v)))], axis=1), [2.5, 97.5])
        ax.plot([lo, hi], [y + off] * 2, color=BMP_COLOR[bm], linewidth=1.8, solid_capstyle="round", zorder=2)
        ax.scatter(np.median(v), y + off, s=60, color=BMP_COLOR[bm], marker=marker[cr], edgecolor="white", linewidth=0.9, zorder=3)
        rows.append([bm, cr, np.median(v), lo, hi, len(v)])

ax.set_yticks(range(len(order)))
ax.set_yticklabels([label[b] for b in order[::-1]], fontsize=10)
ax.tick_params(axis="y", left=False)
ax.tick_params(labelsize=9)
ax.set_xlabel("Change in TN removal efficiency, pre-plant to post-harvest (pp)", fontsize=10)
ax.spines[["top", "right", "left"]].set_visible(False)
ax.legend(handles=[plt.Line2D([], [], marker=marker[c], linestyle="none", markersize=7, color="#555555", label=c.capitalize()) for c in CROPS],
          frameon=False, fontsize=8.5, loc="lower center", ncol=3, bbox_to_anchor=(0.5, -0.3))
fig.tight_layout()
fig.savefig(FIG + "si/phase_pre_to_post.png", dpi=300, bbox_inches="tight")
plt.show()
pd.DataFrame(rows, columns=["bmp", "crop", "median", "ci_lo", "ci_hi", "n"]).round(2).to_csv(TAB + "fig_phase_pre_to_post.csv", index=False)

### Load redistribution
Share of the 2021-2090 baseline TN load falling in each phase, by period and RCP.

In [ ]:
d = load_phase()
d = d[d.scenario != "historical"].drop_duplicates(["crop", "region", "scenario", "phase_year", "phase"]).copy()
d["period"] = np.where(d.phase_year <= EARLY[1], "2021–50", "2051–90")

phases = ["preplant", "midseason", "postharvest"]
phase_color = {"preplant": "#8da0cb", "midseason": "#66c2a5", "postharvest": "#fc8d62"}
bars = [("RCP45", "2021–50"), ("RCP45", "2051–90"), ("RCP85", "2021–50"), ("RCP85", "2051–90")]
x = [0, 0.85, 2.15, 3.0]

fig, axes = plt.subplots(1, 3, figsize=(6.5, 4.4), sharey=True)
for ax, cr in zip(axes, CROPS):
    tot = d[d.crop == cr].groupby(["scenario", "period", "phase"]).baseline_TN.sum().unstack()
    share = tot.div(tot.sum(axis=1), axis=0) * 100
    bottom = np.zeros(len(bars))
    for ph in phases:
        vals = np.array([share.loc[b, ph] for b in bars])
        ax.bar(x, vals, bottom=bottom, width=0.75, color=phase_color[ph], edgecolor="white", linewidth=1.0, label=ph.capitalize())
        for xi, v, bo in zip(x, vals, bottom):
            if v >= 7:
                ax.text(xi, bo + v / 2, f"{v:.0f}", ha="center", va="center", fontsize=8.5, color="white", fontweight="bold")
        bottom += vals
    ax.set_xticks(x)
    ax.set_xticklabels([b[1] for b in bars], fontsize=8, rotation=30, ha="right")
    ax.set_xlim(-0.6, 3.6)
    ax.set_ylim(0, 100)
    for xc, lab in [(0.42, "RCP 4.5"), (2.58, "RCP 8.5")]:
        ax.text(xc, -0.30, lab, transform=ax.get_xaxis_transform(), ha="center", fontsize=9, color="#555555")
    ax.set_title(cr.capitalize(), fontsize=11)
    ax.tick_params(labelsize=9)
    ax.tick_params(axis="x", bottom=False)
    ax.spines[["top", "right"]].set_visible(False)
    print(cr)
    print(share[phases].round(1).to_string())

axes[0].set_ylabel("Share of annual TN load (%)", fontsize=10)
h, l = axes[0].get_legend_handles_labels()
axes[1].legend(h, l, frameon=False, fontsize=8.5, loc="upper center", bbox_to_anchor=(0.5, -0.36), ncol=3)
fig.tight_layout()
plt.subplots_adjust(wspace=0.15)
plt.savefig(FIG + "phase_load_share.png", dpi=300, bbox_inches="tight")
plt.show()

### Change by phase under future climate (Figure 4)
Median TN removal efficiency in the historical period (open) and late century (filled), regions pooled. Asterisks mark Mann-Whitney p < 0.05.

In [ ]:
d = load_phase()
d = d[np.isfinite(d.re_TN)]
hist = d[d.scenario == "historical"]

order = ["nt", "gb", "ma", "cc", "nm"]
phases = ["preplant", "midseason", "postharvest"]
name = {"cc": "Cover\ncrop", "gb": "Grass\nbuffer", "ma": "Manure\nincorporation", "nm": "Nutrient\nmanagement", "nt": "No-till"}

rows = []
for cr in CROPS:
    for bm in order:
        for ph in phases:
            h = hist[(hist.crop == cr) & (hist.bmp == bm) & (hist.phase == ph)].re_TN
            if len(h) == 0:
                continue
            row = [cr, bm, ph, h.median()]
            for sc in ["RCP45", "RCP85"]:
                l = d[(d.scenario == sc) & d.phase_year.between(*LATE) & (d.crop == cr) & (d.bmp == bm) & (d.phase == ph)].re_TN
                row += [l.median(), mannwhitneyu(h, l).pvalue] if len(l) else [np.nan, np.nan]
            rows.append(row)
s = pd.DataFrame(rows, columns=["crop", "bmp", "phase", "hist", "late45", "p45", "late85", "p85"])
s.round(3).to_csv(TAB + "fig_phase_change.csv", index=False)
s = s.set_index(["crop", "bmp", "phase"])

fig, axes = plt.subplots(1, 3, figsize=(7, 4), sharey=True)
for ax, cr in zip(axes, CROPS):
    for i, bm in enumerate(order):
        if i % 2 == 0:
            ax.axhspan(i * 3 - 0.5, i * 3 + 2.5, color="0.95", zorder=0)
        for k, ph in enumerate(phases):
            if (cr, bm, ph) not in s.index:
                continue
            r = s.loc[(cr, bm, ph)]
            for col, pcol, mk, dy in [("late45", "p45", "o", -0.12), ("late85", "p85", "^", 0.12)]:
                if np.isnan(r[col]):
                    continue
                y = i * 3 + k + dy
                ax.plot([r["hist"], r[col]], [y, y], color=BMP_COLOR[bm], lw=1.2, zorder=2)
                ax.plot(r["hist"], y, mk, ms=4, mfc="white", mec=BMP_COLOR[bm], mew=1.0, zorder=3)
                ax.plot(r[col], y, mk, ms=4, mfc=BMP_COLOR[bm], mec=BMP_COLOR[bm], zorder=3)
                if r[pcol] < 0.05:
                    ax.text(80, y, "*", ha="center", va="center", fontsize=11, weight="bold")
    ax.axvline(0, color="0.3", lw=0.6)
    ax.set_title(cr.capitalize(), fontsize=11)
    ax.spines[["top", "right", "left"]].set_visible(False)
    ax.tick_params(axis="y", length=0)
    ax.tick_params(axis="x", labelsize=10)
    ax.set_xlim(-45, 90)

axes[0].set_yticks(range(15))
axes[0].set_yticklabels(["Pre", "Mid", "Post"] * 5, fontsize=9)
axes[0].set_ylim(14.5, -0.5)
for i, bm in enumerate(order):
    axes[0].text(-0.2, i * 3 + 1, name[bm], fontsize=10, ha="right", va="center", transform=axes[0].get_yaxis_transform())
axes[1].set_xlabel("Median TN removal efficiency (%)", fontsize=11)

handles = [
    plt.Line2D([], [], color="0.3", marker="o", ms=5, mfc="white", mec="0.3", ls="none", label="Historical (1991–2000)"),
    plt.Line2D([], [], color="0.3", marker="o", ms=5, mfc="0.3", mec="0.3", ls="-", label="RCP 4.5 (2051–2090)"),
    plt.Line2D([], [], color="0.3", marker="^", ms=5, mfc="0.3", mec="0.3", ls="-", label="RCP 8.5 (2051–2090)"),
    plt.Line2D([], [], color="none", marker="$*$", ms=7, mfc="black", mec="black", ls="none", label="p < 0.05"),
]
fig.legend(handles=handles, loc="lower center", bbox_to_anchor=(0.5, 0.02), ncol=4, frameon=False, fontsize=9)
fig.tight_layout(rect=(0, 0.10, 1, 1))
fig.savefig(FIG + "phase_change.png", dpi=300)
plt.show()

## Storm events

### Storm size and load (SI)
Baseline TN load by storm runoff quartile (median and IQR), and the share of total baseline TN load from each quartile.

In [ ]:
d = load_storms()
q_labels = ["Q1\nlow", "Q2", "Q3", "Q4\nhigh"]
d["q"] = pd.qcut(d.event_runoff_Q, 4, labels=q_labels)

fig, (ax0, ax1) = plt.subplots(1, 2, figsize=(6.5, 4))
q = d.groupby("q", observed=True).baseline_TN.quantile([0.25, 0.5, 0.75]).unstack()
ax0.fill_between(range(4), q[0.25], q[0.75], alpha=0.2, color="black", label="IQR")
ax0.plot(range(4), q[0.5], marker="o", color="black", lw=2, label="Median")
ax0.set_ylabel("Baseline TN Load (kg/ha)", fontsize=12)
ax0.legend(loc="upper left", frameon=False, fontsize=10)
ax0.set_title("A.", fontsize=10)

storms = d.drop_duplicates(["scenario", "region", "crop", "start"]).copy()
storms["q"] = pd.qcut(storms.event_runoff_Q, 4, labels=q_labels)
share = storms.groupby("q", observed=True).baseline_TN.sum()
share = share / share.sum() * 100
ax1.bar(range(4), share.values, color="#555555", width=0.7)
for i, v in enumerate(share.values):
    ax1.text(i, v + 1, f"{v:.0f}", ha="center", va="bottom", fontsize=9, color="#333")
ax1.set_ylabel("Share of Total TN Load (%)", fontsize=12)
ax1.set_ylim(0, share.max() * 1.18)
ax1.set_title("B.", fontsize=10)

for ax in [ax0, ax1]:
    ax.set_xticks(range(4))
    ax.set_xticklabels(q_labels, fontsize=11)
    ax.set_xlabel("Storm Runoff Quartile", fontsize=12)
    ax.spines[["top", "right"]].set_visible(False)
    ax.grid(axis="y", alpha=0.25, ls=":")
plt.tight_layout()
plt.savefig(FIG + "si/storm_load_by_quartile.png", dpi=300, bbox_inches="tight")
plt.show()

### Storm removal efficiency by scenario (SI)
Median storm TN removal efficiency with bootstrapped 95% CIs.

In [ ]:
d = load_storms()
rng = np.random.default_rng(0)
scen = ["historical", "RCP45", "RCP85"]
order = ["gb", "nt", "ma", "nm", "cc"]
color = {"gb": "#253494", "nt": "#2c7fb8", "ma": "#41b6c4", "nm": "#e6810a", "cc": "#c0392b"}
label = {"gb": "Grass buffer", "nt": "No-till", "ma": "Manure incorp.", "nm": "Nutrient mgmt.", "cc": "Cover crop"}

fig, ax = plt.subplots(figsize=(6, 4))
ax.axhline(0, color="#888", alpha=0.5, lw=1, ls="--", zorder=1)
for bm in order:
    med, lo, hi = [], [], []
    for sc in scen:
        v = d[(d.bmp == bm) & (d.scenario == sc)].re_TN.dropna().values
        draws = np.median(v[rng.integers(0, len(v), (N_BOOT, len(v)))], axis=1)
        med.append(np.median(v))
        lo.append(np.median(v) - np.percentile(draws, 2.5))
        hi.append(np.percentile(draws, 97.5) - np.median(v))
    ax.errorbar(range(3), med, yerr=[lo, hi], color=color[bm], lw=2.0, marker="o", ms=6, capsize=3, capthick=1,
                elinewidth=1, zorder=3, label=label[bm])

ax.set_xticks(range(3))
ax.set_xticklabels(["Historical", "RCP 4.5", "RCP 8.5"], fontsize=11)
ax.set_ylabel("Median storm-scale\nTN removal efficiency (%)", fontsize=12)
ax.set_xlim(-0.25, 2.55)
ax.annotate("Transport\nControl", xy=(0.92, 0.39), xycoords="axes fraction", fontsize=9, ha="center", color="#2A82C1")
ax.annotate("Source\nReduction", xy=(0.92, 0.05), xycoords="axes fraction", fontsize=9, ha="center", color="#BF6600")
ax.annotate("Hybrid", xy=(0.92, 0.85), xycoords="axes fraction", fontsize=9, ha="center", color="#A10000")
ax.legend(frameon=False, fontsize=10, loc="lower center", bbox_to_anchor=(0.5, -0.3), ncol=3)
ax.spines[["top", "right"]].set_visible(False)
ax.grid(axis="y", alpha=0.25, lw=0.6)
fig.tight_layout()
fig.savefig(FIG + "si/storm_re_by_scenario.png", dpi=300, bbox_inches="tight")
plt.show()

### Storm size vs. timing (Figure 5)
A: observed storms (historical and 2021-2090, both RCPs) grouped by runoff quartile. B: synthetic storms grouped by crop-cycle phase. Crops, regions, and scenarios pooled; medians with bootstrapped 95% CIs.

In [ ]:
storms = load_storms()
storms["level"] = pd.qcut(storms.event_runoff_Q, 4, labels=["Q1", "Q2", "Q3", "Q4"]).astype(str)
storms["re"] = storms.re_TN

syn = pd.concat([pd.read_csv(DATA + f).assign(crop=c) for c, f in SYN_FILES.items()])
syn = syn[syn.bmp.isin(BMPS) & np.isfinite(syn.RE_TN) & (syn.base_Q > 0)].copy()
syn["level"] = syn.window
syn["re"] = syn.RE_TN * 100

rng = np.random.default_rng(0)
label = {"cc": "Cover Crop", "gb": "Grass Buffer", "ma": "Manure Incorporation", "nm": "Nutrient Management", "nt": "No-Till"}
panels = [(storms, ["Q1", "Q2", "Q3", "Q4"], ["Q1\nlow", "Q2", "Q3", "Q4\nhigh"], "Storm runoff quartile", "magnitude"),
          (syn, ["preplant", "midseason", "postharvest"], ["Pre-\nplant", "Mid-\nseason", "Post-\nharvest"], "Position in the crop cycle", "timing")]

fig, axes = plt.subplots(1, 2, figsize=(6.5, 3), sharey=True, gridspec_kw={"wspace": 0.12})
rows = []
for ax, (df, levels, ticks, xlabel, tag) in zip(axes, panels):
    for bm in ["cc", "gb", "nt", "ma", "nm"]:
        xs, med, lo, hi = [], [], [], []
        for i, lv in enumerate(levels):
            v = df[(df.bmp == bm) & (df.level == lv)].re.dropna().values
            if len(v) < 20:
                continue
            draws = np.median(v[rng.integers(0, len(v), (N_BOOT, len(v)))], axis=1)
            xs.append(i)
            med.append(np.median(v))
            lo.append(np.percentile(draws, 2.5))
            hi.append(np.percentile(draws, 97.5))
            rows.append([tag, bm, lv, med[-1], lo[-1], hi[-1], len(v)])
        ax.fill_between(xs, lo, hi, color=BMP_COLOR[bm], alpha=0.22, lw=0)
        ax.plot(xs, med, marker="o", ms=5, color=BMP_COLOR[bm], lw=1.6, label=label[bm])
    ax.set_xticks(range(len(levels)))
    ax.set_xticklabels(ticks, fontsize=9)
    ax.set_xlabel(xlabel, fontsize=10)
    ax.axhline(0, color="#CCCCCC", lw=0.8, zorder=0)
    ax.spines[["top", "right"]].set_visible(False)
    ax.tick_params(labelsize=8)

axes[0].set_ylabel("TN removal efficiency (%)", fontsize=10)
axes[0].set_title("A.", fontweight="bold", fontsize=11, loc="left")
axes[1].set_title("B.", fontweight="bold", fontsize=11, loc="left")
axes[1].legend(frameon=False, fontsize=7.6, loc="lower center", bbox_to_anchor=(-0.1, -0.4), ncol=5, handlelength=1.2, columnspacing=1.0)
fig.savefig(FIG + "storm_size_vs_timing.png", dpi=400, bbox_inches="tight")
plt.show()
pd.DataFrame(rows, columns=["panel", "bmp", "level", "median_re", "ci_lo", "ci_hi", "n"]).round(1).to_csv(TAB + "fig_storm_size_vs_timing.csv", index=False)

## Synthetic storm experiment

### Storm timing by crop
Median TN removal efficiency by crop-cycle phase for each crop, all warming levels pooled; bands are bootstrapped 95% CIs.

In [ ]:
s = pd.concat([pd.read_csv(DATA + f).assign(crop=c) for c, f in SYN_FILES.items()])
s = s[s.bmp.isin(BMPS)].dropna(subset=["RE_TN"])
rng = np.random.default_rng(0)

windows = ["preplant", "midseason", "postharvest"]
name = {"cc": "Cover Crop", "gb": "Grass Buffer", "ma": "Manure\nIncorporation", "nm": "Nutrient\nManagement", "nt": "No-Tillage"}

fig, axes = plt.subplots(1, 3, figsize=(6.5, 3), sharey=True)
for ax, cr in zip(axes, CROPS):
    for bm in BMPS:
        med, lo, hi = [], [], []
        for w in windows:
            v = s[(s.crop == cr) & (s.bmp == bm) & (s.window == w)].RE_TN.values * 100
            if len(v) < 30:
                med.append(np.nan)
                lo.append(np.nan)
                hi.append(np.nan)
                continue
            draws = np.median(v[rng.integers(0, len(v), (N_BOOT, len(v)))], axis=1)
            med.append(np.median(v))
            lo.append(np.percentile(draws, 2.5))
            hi.append(np.percentile(draws, 97.5))
        if np.all(np.isnan(med)):
            continue
        ax.fill_between([0, 1, 2], lo, hi, color=BMP_COLOR[bm], alpha=0.15, lw=0)
        ax.plot([0, 1, 2], med, marker="o", ms=5, lw=1.9, color=BMP_COLOR[bm], mec="white", mew=0.7, zorder=4 if bm == "nt" else 3)
    ax.axhline(0, color="#CCCCCC", lw=0.8, zorder=0)
    ax.set_xticks([0, 1, 2])
    ax.set_xticklabels(["Pre", "Mid", "Post"], fontsize=9)
    ax.set_xlim(-0.2, 2.2)
    ax.set_title(cr.capitalize(), fontsize=11, pad=5)
    ax.spines[["top", "right"]].set_visible(False)
    ax.spines[["left", "bottom"]].set_linewidth(0.8)
    ax.tick_params(labelsize=9)

axes[0].set_ylim(-60, 75)
axes[0].set_ylabel("TN removal efficiency (%)", fontsize=9)
fig.legend(handles=[plt.Line2D([], [], color=BMP_COLOR[b], lw=2, marker="o", ms=5, mec="white", label=name[b]) for b in BMPS],
           frameon=False, fontsize=8.5, ncol=5, loc="lower center", bbox_to_anchor=(0.5, -0.15), columnspacing=1.2, handletextpad=0.4)
fig.text(0.5, 0.005, "Position in the crop cycle when the storm arrives", ha="center", fontsize=9)
plt.tight_layout(rect=[0, 0.05, 1, 1])
plt.savefig(FIG + "storm_timing_by_crop.png", dpi=400, bbox_inches="tight")
plt.show()

### Warming effect on field state and removal efficiency (Figure 6)
A: change in crop development (HUI) at storm time relative to the unperturbed run. B: absolute change in TN removal efficiency for the same storm. Binned by 0.5 °C of annual mean warming; lines are medians and bands the IQR. Only storms with defined removal efficiency in at least 7 of the 12 scenarios are used in B.

In [ ]:
storm = ["region", "window", "date_idx", "return_period"]
crop_color = {"Corn": "#214f5e", "Soy": "#4c9a8e", "Wheat": "#a7c785"}

hui_all, re_all = [], []
for cr, f in SYN_FILES.items():
    d = pd.read_csv(DATA + f)
    d["crop"] = cr.capitalize()

    hui = d[d.bmp == "base"].copy()
    ref = hui[hui.scenario == "baseline"].set_index(storm).HUI.rename("HUI_ref")
    hui = hui[hui.scenario != "baseline"].join(ref, on=storm)
    hui = hui[(hui.HUI > 0) & (hui.HUI_ref > 0)]
    hui["change"] = hui.HUI - hui.HUI_ref
    hui_all.append(hui)

    re = d[d.bmp.isin(BMPS)].dropna(subset=["RE_TN"]).copy()
    re = re[re.groupby(storm + ["bmp"]).RE_TN.transform("size") >= 7]
    ref = re[re.scenario == "baseline"].set_index(storm + ["bmp"]).RE_TN.rename("RE_ref")
    re = re[re.scenario != "baseline"].join(ref, on=storm + ["bmp"]).dropna(subset=["RE_ref"])
    re["change"] = (re.RE_TN - re.RE_ref).abs() * 100
    re_all.append(re)

fig, axes = plt.subplots(1, 2, figsize=(7, 3.5), sharex=True)
for ax, d, tag in [(axes[0], pd.concat(hui_all), "HUI"), (axes[1], pd.concat(re_all), "RE")]:
    d["bin"] = (d.annual_mean_dT * 2).round() / 2
    stats = d.groupby(["crop", "bin"]).change.quantile([0.25, 0.5, 0.75]).unstack()
    stats.round(3).to_csv(TAB + f"fig_warming_effect_{tag}.csv")
    for cr, c in crop_color.items():
        st = stats.loc[cr]
        ax.fill_between(st.index, st[0.25], st[0.75], color=c, alpha=0.12, lw=0)
        ax.plot(st.index, st[0.5], color=c, lw=2.2, marker="o", ms=3.5, label=cr)
        ax.plot([0, st.index[0]], [0, st[0.5].iloc[0]], color=c, lw=1, ls=":")
    ax.plot(0, 0, "o", color="0.3", ms=5, zorder=5)
    ax.set_xlabel("Annual mean temperature change (°C)", fontsize=10)
    ax.spines[["top", "right"]].set_visible(False)
    ax.set_ylim(bottom=0)

axes[0].set_ylabel("Change in crop development (HUI)", fontsize=10)
axes[1].set_ylabel("Absolute change in TN removal efficiency (pp)", fontsize=10)
axes[0].set_title("A.", loc="left", fontweight="bold", fontsize=11.5)
axes[1].set_title("B.", loc="left", fontweight="bold", fontsize=11.5)
axes[0].legend(frameon=False, fontsize=10, loc="upper left")
plt.tight_layout()
plt.savefig(FIG + "warming_effect.png", dpi=300, bbox_inches="tight")
plt.show()



### Warming effect by crop and storm timing (Figure 7)
Median TN removal efficiency for each crop and phase under the unperturbed climate, +2.0 °C (RCP 4.5, 2041-2070), and +4.7 °C (RCP 8.5, 2070-2099).

In [ ]:
s = pd.concat([pd.read_csv(DATA + f).assign(crop=c) for c, f in SYN_FILES.items()])
levels = ["baseline", "RCP45_2041_2070", "RCP85_2070_2099"]
level_name = {"baseline": "Historical", "RCP45_2041_2070": "+2.0 °C", "RCP85_2070_2099": "+4.7 °C"}
level_color = {"baseline": "0.55", "RCP45_2041_2070": "#F4A261", "RCP85_2070_2099": "#B2182B"}
name = {"cc": "Cover crop", "gb": "Grass buffer", "ma": "Manure\nincorporation", "nm": "Nutrient\nmanagement", "nt": "No-till"}
marker = {"corn": "o", "soy": "s", "wheat": "^"}
phases = ["preplant", "midseason", "postharvest"]

s = s[s.bmp.isin(BMPS) & s.scenario.isin(levels)]
g = s.groupby(["crop", "region", "window", "date_idx", "bmp", "scenario"]).RE_TN.median().unstack("scenario").dropna() * 100
m = g.groupby(["bmp", "window", "crop"]).median()[levels]
m.round(1).to_csv(TAB + "fig_warming_by_timing.csv")

fig, axes = plt.subplots(1, 5, figsize=(7.0, 3.5), sharey=True)
for ax, bm in zip(axes, BMPS):
    ax.axhline(0, color="0.3", lw=0.5, ls=":")
    for i, ph in enumerate(phases):
        for xi, cr in zip([i - 0.25, i, i + 0.25], CROPS):
            if (bm, ph, cr) not in m.index:
                continue
            y = m.loc[(bm, ph, cr)].values
            ax.plot([xi] * 3, y, color="0.75", lw=0.8, zorder=1)
            for yi, lv in zip(y, levels):
                ax.plot(xi, yi, marker=marker[cr], color=level_color[lv], ms=4.5, ls="none", zorder=3)
    ax.set_xticks(range(3))
    ax.set_xticklabels(["Pre-\nPlant", "Mid-\nSeason", "Post-\nHarvest"], rotation=45, ha="right", fontsize=8)
    ax.set_xlim(-0.5, 2.5)
    ax.set_title(name[bm], fontsize=8)
    ax.tick_params(axis="y", labelsize=8)
    ax.spines[["top", "right"]].set_visible(False)
axes[0].set_ylim(-60, 100)
axes[0].set_ylabel("TN removal efficiency (%)", fontsize=8)
fig.supxlabel("Storm timing relative to crop cycle", fontsize=8, y=0.23)

handles = [plt.Line2D([], [], color=level_color[lv], marker="o", ms=5, ls="none", label=level_name[lv]) for lv in levels]
handles += [plt.Line2D([], [], color="0.4", marker=marker[c], ms=5, ls="none", label=c.capitalize()) for c in CROPS]
fig.legend(handles=handles, loc="lower center", bbox_to_anchor=(0.5, 0.1), ncol=6, frameon=False, columnspacing=1.2, fontsize=8)
fig.tight_layout(rect=(0, 0.18, 1, 1))
fig.savefig(FIG + "warming_by_timing.png", dpi=300)
plt.show()

## Removal pathways (Figure 8)
Share of nutrient mass removed through soluble and sediment-bound pathways, all periods. Numbers above the bars are median annual removal efficiency.

In [ ]:
d = load_annual()
key = ["scenario", "crop", "region", "year"]
cols = ["QN", "YN", "QP", "YP"]

rows = []
for bm in BMPS:
    ref = d[d.bmp == ("ma-base" if bm == "ma" else "base")].set_index(key)[cols]
    j = d[d.bmp == bm].set_index(key)[cols].join(ref, rsuffix="_b", how="inner")
    removed = {c: max((j[c + "_b"] - j[c]).sum(), 0) for c in cols}
    re_tn = 100 * (1 - (j.QN + j.YN) / (j.QN_b + j.YN_b).replace(0, np.nan))
    re_tp = 100 * (1 - (j.QP + j.YP) / (j.QP_b + j.YP_b).replace(0, np.nan))
    rows.append([bm, 100 * removed["QN"] / (removed["QN"] + removed["YN"]), 100 * removed["QP"] / (removed["QP"] + removed["YP"]),
                 re_tn.median(), re_tp.median()])
v = pd.DataFrame(rows, columns=["bmp", "sol_N", "sol_P", "re_TN", "re_TP"]).set_index("bmp")
v.round(1).to_csv(TAB + "fig_removal_pathways.csv")

order = ["gb", "nt", "ma", "nm", "cc"]
label = {"gb": "Grass Buffer", "nt": "No-Till", "nm": "Nutrient\nManagement", "ma": "Manure\nIncorporation", "cc": "Cover Crop"}
c_sol_n, c_sed_n, c_sol_p, c_sed_p = "#56B4E9", "#0072B2", "#E69F00", "#D55E00"
centers = np.arange(len(order)) * 2.2

fig, ax = plt.subplots(figsize=(6.5, 3.5))
for gc, bm in zip(centers, order):
    for x, sol, re, c_sol, c_sed, tag in [(gc - 0.45, v.loc[bm, "sol_N"], v.loc[bm, "re_TN"], c_sol_n, c_sed_n, "TN"),
                                          (gc + 0.45, v.loc[bm, "sol_P"], v.loc[bm, "re_TP"], c_sol_p, c_sed_p, "TP")]:
        ax.bar(x, sol, 0.7, color=c_sol, zorder=3)
        ax.bar(x, 100 - sol, 0.7, bottom=sol, color=c_sed, zorder=3)
        ax.text(x, 102, f"{re:.0f}%", ha="center", va="bottom", fontsize=8)
        ax.text(x, -4, tag, ha="center", va="top", fontsize=8)

bracket = dict(arrowprops=dict(arrowstyle="-", color="black", lw=0.25), annotation_clip=False)
ax.annotate("", xy=(centers[0] - 0.8, 116), xytext=(centers[2] + 0.8, 116), **bracket)
ax.text(centers[:3].mean(), 118, "Transport control", ha="center", va="bottom", fontsize=8.5)
ax.text(centers[3], 118, "Source reduction", ha="center", va="bottom", fontsize=8.5)
ax.text(centers[4], 118, "Hybrid", ha="center", va="bottom", fontsize=8.5)

ax.set_xticks(centers)
ax.set_xticklabels([label[b] for b in order], fontsize=9)
ax.tick_params(axis="x", pad=18)
ax.set_ylabel("Share of nutrient mass\nremoved (%)", fontsize=10)
ax.set_ylim(0, 128)
ax.set_yticks([0, 25, 50, 75, 100])
ax.spines[["top", "right"]].set_visible(False)
ax.legend(handles=[mpatches.Patch(color=c_sol_n, label="Soluble N"), mpatches.Patch(color=c_sed_n, label="Sediment-bound N"),
                   mpatches.Patch(color=c_sol_p, label="Soluble P"), mpatches.Patch(color=c_sed_p, label="Sediment-bound P")],
          loc="lower center", bbox_to_anchor=(0.5, -0.5), frameon=False, fontsize=8, ncol=4)
plt.tight_layout()
plt.savefig(FIG + "removal_pathways.png", dpi=300, bbox_inches="tight")
plt.show()

## Spin-up length (SI)
Mean loads and removal efficiency for 10-, 20-, 30-, and 40-year spin-ups over the historical period.

In [ ]:
files = {"10yr": "spinup/spinup_10yr.csv", "20yr": "spinup/spinup_20yr.csv", "30yr": "spinup/spinup_30yr.csv", "40yr": "spinup/spinup_40yr.csv"}
loads = ["TN", "TP", "MUSL"]
re_metrics = ["RE_TN", "RE_TP", "RE_MUSL"]
load_label = {"TN": "TN (kg/ha)", "TP": "TP (kg/ha)", "MUSL": "Sediment (t/ha)"}

base_mean = {m: [] for m in loads}
bmp_mean = {m: [] for m in loads}
re_mean = {m: [] for m in re_metrics}
for f in files.values():
    d = pd.read_csv(DATA + f)
    d = d[(d.YLDG > 0) | (d.YLDF > 0)]
    for m in loads:
        base_mean[m].append(d[d.bmp == "base"][m].mean())
        bmp_mean[m].append(d[d.bmp != "base"][m].mean())
    for m in re_metrics:
        v = pd.to_numeric(d[m], errors="coerce").replace([np.inf, -np.inf], np.nan)
        re_mean[m].append(v[v.abs() < 200].mean())

x = np.arange(len(files))
fig, axes = plt.subplots(2, 3, figsize=(6.5, 4), sharex="col", sharey="row")
for i, m in enumerate(loads):
    axes[0, i].bar(x - 0.175, base_mean[m], 0.35, color="#8da0cb", edgecolor="black", linewidth=0.5)
    axes[0, i].bar(x + 0.175, bmp_mean[m], 0.35, color="#fc8d62", edgecolor="black", linewidth=0.5)
    axes[0, i].set_title(load_label[m], fontsize=11)
    axes[1, i].bar(x, re_mean[re_metrics[i]], color="#66c2a5", edgecolor="black", linewidth=0.5)
    axes[1, i].set_xlabel("Spinup Length", fontsize=9)
    for ax in axes[:, i]:
        ax.set_xticks(x)
        ax.set_xticklabels(list(files), rotation=45, ha="right", fontsize=8)
axes[0, 0].set_ylabel("Mean Loading", fontsize=9)
axes[1, 0].set_ylabel("Mean Removal (%)", fontsize=9)

fig.legend(handles=[mpatches.Patch(facecolor="#8da0cb", edgecolor="black", lw=0.5, label="Baseline Load"),
                    mpatches.Patch(facecolor="#fc8d62", edgecolor="black", lw=0.5, label="BMP Load (aggregated)"),
                    mpatches.Patch(facecolor="#66c2a5", edgecolor="black", lw=0.5, label="Mean Removal (aggregated)")],
           frameon=False, loc="lower center", ncol=3, fontsize=8, bbox_to_anchor=(0.5, -0.05))
plt.tight_layout()
plt.savefig(FIG + "si/spinup_loads_and_re.png", dpi=300, bbox_inches="tight")
plt.show()

## CBP technical report

### Annual removal efficiency by region
One figure per region: 15-year rolling median of TN, TP, and sediment removal efficiency for each crop (alfalfa included), with the historical median as a bar over 1991-2000.

In [ ]:
d = load_annual(CROPS + ["alfalfa"])
d = d[d.bmp.isin(BMPS)]
crops = ["corn", "soy", "wheat", "alfalfa"]
regions = {"a": "Appalachian Plateau", "cp": "Coastal Plain", "p": "Piedmont", "vr": "Valley and Ridge"}
name = {"cc": "Cover crop", "gb": "Grass buffer", "ma": "Manure incorporation", "nm": "Nutrient management", "nt": "No-till"}
pollutants = {"RE_TN": "TN", "RE_TP": "TP", "RE_MUSL": "Sediment"}

for rg, rg_name in regions.items():
    fig, axes = plt.subplots(len(crops), 3, figsize=(7.5, 9), sharex=True)
    for i, cr in enumerate(crops):
        for j, (col, pname) in enumerate(pollutants.items()):
            ax = axes[i, j]
            for bm in BMPS:
                g = d[(d.region == rg) & (d.crop == cr) & (d.bmp == bm)]
                for sc, ls in [("RCP 45", "-"), ("RCP 85", "--")]:
                    f = g[g.scenario == sc].set_index("year")[col]
                    if len(f):
                        f = f.reindex(range(EARLY[0], LATE[1] + 1)).rolling(15, center=True, min_periods=7).median()
                        ax.plot(f.index, f.values, color=BMP_COLOR[bm], ls=ls, lw=1.3)
                h = g[g.scenario == "reference period"][col]
                if len(h):
                    ax.plot(HIST, [h.median()] * 2, color=BMP_COLOR[bm], lw=3, solid_capstyle="butt")
            ax.axhline(0, color="0.4", lw=0.5)
            ax.axvline(2010, color="0.8", lw=0.6)
            ax.spines[["top", "right"]].set_visible(False)
            ax.tick_params(labelsize=8)
            if i == 0:
                ax.set_title(pname + " removal efficiency (%)", fontsize=8)
            if j == 0:
                ax.set_ylabel(cr.capitalize(), fontsize=8)
    handles = [plt.Line2D([], [], color=BMP_COLOR[b], lw=1.5, label=name[b]) for b in BMPS]
    handles += [plt.Line2D([], [], color="0.3", ls="-", label="RCP 4.5"), plt.Line2D([], [], color="0.3", ls="--", label="RCP 8.5")]
    fig.legend(handles=handles, loc="lower center", ncol=4, frameon=False, fontsize=8)
    fig.suptitle(rg_name, fontsize=10)
    fig.tight_layout(rect=(0, 0.06, 1, 0.97))
    fig.savefig(FIG + f"cbp/annual_re_{rg}.png", dpi=200)
    plt.show()

### Annual baseline loads by region
15-year rolling median of baseline loads with bootstrapped 95% CIs. RCP 8.5 is drawn from 2051.

In [ ]:
d = load_annual(CROPS + ["alfalfa"])
d = d[d.bmp == "base"]
rng = np.random.default_rng(0)
crops = ["corn", "soy", "wheat", "alfalfa"]
region_name = {"a": "Appalachian Plateau", "cp": "Coastal Plain", "p": "Piedmont", "vr": "Valley and Ridge"}
region_color = {"a": "#228833", "cp": "#4477AA", "p": "#CCBB44", "vr": "#AA3377"}
loads = {"TN": "TN load (kg/ha)", "TP": "TP load (kg/ha)", "MUSL": "Sediment load (t/ha)"}


def median_ci(x):
    draws = np.median(x[rng.integers(0, len(x), (N_BOOT, len(x)))], axis=1)
    return np.median(x), np.percentile(draws, 2.5), np.percentile(draws, 97.5)


fig, axes = plt.subplots(len(crops), 3, figsize=(7.5, 9), sharex=True)
for i, cr in enumerate(crops):
    for j, (col, ylabel) in enumerate(loads.items()):
        ax = axes[i, j]
        for rg, c in region_color.items():
            sel = d[(d.region == rg) & (d.crop == cr)]
            for sc, ls, start in [("RCP 45", "-", EARLY[0]), ("RCP 85", "--", LATE[0])]:
                g = sel[(sel.scenario == sc) & (sel.year >= start)].dropna(subset=[col])
                out = []
                for y in range(start, LATE[1] + 1):
                    x = g[g.year.between(y - 7, y + 7)][col].values
                    if len(x) >= 7:
                        out.append((y,) + median_ci(x))
                out = np.array(out)
                ax.fill_between(out[:, 0], out[:, 2], out[:, 3], color=c, alpha=0.12, lw=0)
                ax.plot(out[:, 0], out[:, 1], color=c, ls=ls, lw=1.3)
            m, lo, hi = median_ci(sel[sel.scenario == "reference period"][col].dropna().values)
            ax.fill_between(HIST, lo, hi, color=c, alpha=0.12, lw=0)
            ax.plot(HIST, [m, m], color=c, lw=3, solid_capstyle="butt")
        ax.axvline(2010, color="0.8", lw=0.6)
        ax.set_ylim(bottom=0)
        ax.spines[["top", "right"]].set_visible(False)
        ax.tick_params(labelsize=8)
        if i == 0:
            ax.set_title(ylabel, fontsize=8)
        if j == 0:
            ax.set_ylabel(cr.capitalize(), fontsize=8)

handles = [plt.Line2D([], [], color=c, lw=1.5, label=region_name[r]) for r, c in region_color.items()]
handles += [plt.Line2D([], [], color="0.3", ls="-", label="RCP 4.5 (from 2021)"), plt.Line2D([], [], color="0.3", ls="--", label="RCP 8.5 (from 2051)"),
            plt.Rectangle((0, 0), 1, 1, color="0.3", alpha=0.2, lw=0, label="95% CI of median")]
fig.legend(handles=handles, loc="lower center", ncol=4, frameon=False, fontsize=8)
fig.tight_layout(rect=(0, 0.07, 1, 1))
fig.savefig(FIG + "cbp/annual_load_by_region.png", dpi=200)
plt.show()